### TOOLS

In [ ]:
from langchain_community.chat_models import ChatOllama

# Use the local tinyllama model that was downloaded successfully
model = ChatOllama(model="tinyllama")

response = model.invoke("Why do parrots talk?")
print(response.content)


ImportError: cannot import name 'ChatOllama' from 'langchain_community.chat_models' (/Users/pratham/LangChainnn/.venv/lib/python3.11/site-packages/langchain_community/chat_models/__init__.py)

: 

### importing tools and binding tools

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
   """Get the weather at a location"""
   return f"It's sunny is {location}"

model_with_tools=model.bind_tools([get_weather])

### Calling the Tools

In [ ]:
# tool call
response =model_with_tools.invoke("What's the weather like in Edinburgh?")
print(response)
for tool_call in response.tool_calls:
   # View tool calls made by the model
   print(f"Tool:{tool_call['name']}")
   print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'v3k707er4', 'function': {'arguments': '{"location":"Edinburgh"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 278, 'total_tokens': 305, 'completion_time': 0.071727392, 'completion_tokens_details': None, 'prompt_time': 0.019525001, 'prompt_tokens_details': None, 'queue_time': 0.047275378, 'total_time': 0.091252393}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0fc8f-a168-76a3-90ea-0c6e2c4aa8f5-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Edinburgh'}, 'id': 'v3k707er4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 27, 'total_tokens': 305}
Tool:get_weather
Args: {'location': 'Edinburgh'}


### Tool Execution Loops

A tool execution loop is the process where an LLM repeatedly decides whether it needs a tool, calls the tool, receives the result, and uses that result to continue generating an answer until the task is completed.

In [ ]:

# Step 1: Model generates tool calls
messages = [{"role":"user","content":" What is the weather like in Edinburhg?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step:2 Execute Tools and collect Results
for tool_call in ai_msg.tool_calls:
   # Execute the tool with the generated arguments
   tool_result = get_weather.invoke(tool_call)
   messages.append(tool_result)

# Step 3: Pass results back to model for final response 
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in edinburgh is 72 F and sunny"

The weather in Edinburgh is currently sunny! ☀️
